# Smriti × official Laya: train a relevance head on a GPU

Round 4 of Smriti's decision-model trials. The frozen **official Laya encoder**
(`convaiinnovations/laya`, 421M ModernBERT-large, pinned revision `55cf4c4`) reads each
(question, memory) pair in Smriti's top-20 pools; a small logistic head is trained on the
**dev** questions and scored once on the **held-out test** questions. Also measures
zero-shot Laya on the full pools and GPU latency.

**How to run:** Runtime → Change runtime type → any GPU (T4 is enough; L4 or A100 is faster)
→ Runtime → **Run all**. Approve the Google sign-in pop-up in step 1. About 15–25 minutes.

**Where results go:** one zip, `smriti-colab-laya-head.zip`, in your Google Drive, shared
view-only by link so the Smriti lab can fetch it. It holds encoder features, head weights,
scores, timings and logs; the texts are public LongMemEval turns. Re-uploaded every 5 minutes.

In [ ]:
# 1 · Clone the lab and connect the results file (one Google sign-in pop-up, then walk away)
import os
import subprocess
import sys
# REV: a branch, tag or full commit SHA. "main" runs the current lab. To repeat the recorded
# round 4 trials exactly, use 2b52da2173e5275da3c13cee8fd3185fc14b18c3 (Laya head) or
# 97a657275b72c6da1328eb7b46d95b77fdbc5756 (CLM-8B, all three modes).
REV = "main"
os.makedirs("/content/Smriti", exist_ok=True)
if not os.path.isdir("/content/Smriti/.git"):
    subprocess.run(["git", "init", "-q"], cwd="/content/Smriti", check=True)
    subprocess.run(["git", "remote", "add", "origin", "https://github.com/vn-envy/Smriti"], cwd="/content/Smriti", check=True)
subprocess.run(["git", "fetch", "-q", "--depth", "1", "origin", REV], cwd="/content/Smriti", check=True)
subprocess.run(["git", "checkout", "-q", "--force", "FETCH_HEAD"], cwd="/content/Smriti", check=True)
os.chdir("/content/Smriti")
COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Smriti lab at", COMMIT)
sys.path.insert(0, "/content/Smriti")
from bench.lab.colab.runner import DriveResults, run  # noqa: E402
POOLS = "audit/2026-09-25/decision-models/pools"

import torch  # noqa: E402
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4, L4 or A100, then Run all."
print("GPU:", torch.cuda.get_device_name(0))
OUT = "/content/laya-head"
os.makedirs(OUT, exist_ok=True)
res = DriveResults("smriti-colab-laya-head.zip")
res.add(OUT)
res.push()

In [ ]:
# 2 · Install Laya (the only extra dependency)
!pip install -q laya==0.3.20

In [ ]:
# 3 · Encode all 10,000 (question, memory) pairs with the frozen official encoder
REV = "55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851"
run(f"python bench/lab/judge_head/extract_pools.py {POOLS} {OUT} --revision {REV}", res)

In [ ]:
# 4 · Train heads on dev (5-fold CV grouped by question), score once on held-out test
run(f"python bench/lab/judge_head/train.py {OUT} | tee {OUT}/train.log", res)

In [ ]:
# 5 · Zero-shot official Laya on the full pools, for reference (GPU)
from huggingface_hub import snapshot_download
MODEL = snapshot_download("convaiinnovations/laya", revision=REV,
                          allow_patterns=["model.safetensors", "rl_agent_config.json", "encoder/*", "tokenizer/*"])
for split in ("dev", "test"):
    run(f"python bench/lab/judge_head/pool_eval.py laya {POOLS}/pools-{split}.jsonl.gz {OUT}/zeroshot-{split}.json "
        f"--workers 1 --probe 30", res,
        env={"SMRITI_LAB_LAYA": MODEL, "SMRITI_LAB_LAYA_DEVICE": "cuda", "SMRITI_LAB_JUDGE_CACHE": f"{OUT}/judge-cache.sqlite"})

In [ ]:
# 6 · Scoreboard (held-out test unless noted)
import json
rep = json.load(open(f"{OUT}/report.json"))
tim = json.load(open(f"{OUT}/timing.json"))
zs = {s: json.load(open(f"{OUT}/zeroshot-{s}.json"))["summary"] for s in ("dev", "test")}
json.dump({"commit": COMMIT, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__},
          open(f"{OUT}/env.json", "w"), indent=1)
rows = [("Smriti's own order", rep["smriti_fused_order"]["test"]["auc"], rep["smriti_fused_order"]["test"]["top1"]),
        ("Official Laya, zero-shot", zs["test"]["auc_judge"], zs["test"]["top1_judge"]),
        ("Head on Smriti signals", rep["smriti"]["test"]["auc"], rep["smriti"]["test"]["top1"]),
        ("Head on official Laya", rep["laya"]["test"]["auc"], rep["laya"]["test"]["top1"]),
        ("Head on Laya + Smriti", rep["both"]["test"]["auc"], rep["both"]["test"]["top1"]),
        ("(ref) community Laya + Smriti head, round 2", 0.928, None),
        ("(ref) hosted Jev, zero-shot, round 3", 0.954, 0.7249)]
print(f"{'Judge':46s} {'AUC':>6s} {'Top-1':>6s}")
for n, a, t in rows:
    print(f"{n:46s} {a:6.3f} {'' if t is None else f'{100*t:5.1f}%':>6s}")
print("\nEncoder on", tim["gpu"], "ms per question (20 pairs):", tim["splits"]["test"]["ms_per_question_p50"])
print("Zero-shot Laya ms per question:", zs["test"]["ms_per_question_p50"])
res.push()
print("\nDone. Tell Claude the run finished (Drive file id above).")